In [27]:
import random
from collections import defaultdict
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from surprise import Dataset, KNNBasic, Reader
from surprise.model_selection import train_test_split


In [ ]:

faceplate_data = pd.read_csv('Faceplate.csv')
print('First 10 faceplate transactions:')
faceplate_data.head(10)


First 10 faceplate transactions:


,Transaction,Red,White,Blue,Orange,Green,Yellow
0,1,1,1,0,0,1,0
1,2,0,1,0,1,0,0
2,3,0,1,1,0,0,0
3,4,1,1,0,1,0,0
4,5,1,0,1,0,0,0
5,6,0,1,1,0,0,0
6,7,1,0,1,0,0,0
7,8,1,1,1,0,1,0
8,9,1,1,1,0,0,0
9,10,0,0,0,0,0,1


In [7]:
print("Qn 1.2: Support of itemset {red, white}")
support_red_white = ((faceplate_data['Red'] == 1) & (faceplate_data['White'] == 1)).mean()
count_red_white = ((faceplate_data['Red'] == 1) & (faceplate_data['White'] == 1)).sum()
total_transactions = len(faceplate_data)
print(f"Support of itemset {{red, white}}: {support_red_white:.2f}")
print(f"({count_red_white} out of {total_transactions} transactions)")


Qn 1.2: Support of itemset {red, white}
Support of itemset {red, white}: 0.40
(4 out of 10 transactions)


In [ ]:

print('Qn 2.1: Frequent itemsets using Apriori (min_support = 0.2)')
faceplate_binary = faceplate_data.drop(columns=['Transaction']).astype(bool)
frequent_itemsets = apriori(faceplate_binary, min_support=0.2, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False).reset_index(drop=True)
frequent_itemsets


Qn 2.1: Frequent itemsets using Apriori (min_support = 0.2)


,support,itemsets
0,0.7,(White)
1,0.6,(Red)
2,0.6,(Blue)
3,0.4,"(Red, White)"
4,0.4,"(Red, Blue)"
5,0.4,"(White, Blue)"
6,0.2,(Orange)
7,0.2,(Green)
8,0.2,"(Green, Red)"
9,0.2,"(Orange, White)"


In [ ]:

print('Qn 2.2: Association rules (min_confidence = 0.5), sorted by lift')
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)
rules_sorted_by_lift = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)
rules_sorted_by_lift


Qn 2.2: Association rules (min_confidence = 0.5), sorted by lift


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"(Red, White)",(Green),0.4,0.2,0.2,0.500000,2.500000,1.0,0.12,1.600000,1.000000,0.500000,0.375000,0.750000
1,(Green),"(Red, White)",0.2,0.4,0.2,1.000000,2.500000,1.0,0.12,inf,0.750000,0.500000,1.000000,0.750000
2,(Green),(Red),0.2,0.6,0.2,1.000000,1.666667,1.0,0.08,inf,0.500000,0.333333,1.000000,0.666667
3,"(Green, White)",(Red),0.2,0.6,0.2,1.000000,1.666667,1.0,0.08,inf,0.500000,0.333333,1.000000,0.666667
4,(Orange),(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
5,(Green),(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
6,"(Green, Red)",(White),0.2,0.7,0.2,1.000000,1.428571,1.0,0.06,inf,0.375000,0.285714,1.000000,0.642857
7,(Red),(Blue),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.200000,0.250000,0.500000,0.166667,0.666667
8,(Blue),(Red),0.6,0.6,0.4,0.666667,1.111111,1.0,0.04,1.200000,0.250000,0.500000,0.166667,0.666667
9,(Red),(White),0.6,0.7,0.4,0.666667,0.952381,1.0,-0.02,0.900000,-0.111111,0.444444,-0.111111,0.619048


In [14]:
# Qn 2.3
print('Qn 2.3: Top 6 rules by lift with requested metrics')
top_6_rules = rules_sorted_by_lift.head(6).copy()
top_6_rules = top_6_rules.drop(columns=['antecedent support', 'consequent support', 'conviction'])
top_6_rules = top_6_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']]
top_6_rules


Qn 2.3: Top 6 rules by lift with requested metrics


,antecedents,consequents,support,confidence,lift,leverage
0,"(Red, White)",(Green),0.2,0.5,2.500000,0.12
1,(Green),"(Red, White)",0.2,1.0,2.500000,0.12
2,(Green),(Red),0.2,1.0,1.666667,0.08
3,"(Green, White)",(Red),0.2,1.0,1.666667,0.08
4,(Orange),(White),0.2,1.0,1.428571,0.06
5,(Green),(White),0.2,1.0,1.428571,0.06


In [15]:

print('Qn 2.4: Interpretation of rule with highest lift ratio')
best_rule = rules_sorted_by_lift.iloc[0]
antecedent_items = ', '.join(sorted(best_rule['antecedents']))
consequent_items = ', '.join(sorted(best_rule['consequents']))
confidence_pct = best_rule['confidence'] * 100
lift_value = best_rule['lift']
sentence = (
    f"If [{antecedent_items}] are purchased, then with confidence {confidence_pct:.1f}% "
    f"[{consequent_items}] will also be purchased. This rule has a lift ratio of {lift_value:.3f}."
)
print(sentence)


Qn 2.4: Interpretation of rule with highest lift ratio
If [Red, White] are purchased, then with confidence 50.0% [Green] will also be purchased. This rule has a lift ratio of 2.500.


In [16]:

print('Qn 3.1: Charles Book Club binary incidence matrix (first 10 rows)')
bookclub_data = pd.read_csv('CharlesBookClub.csv')
book_columns = [
    'ChildBks', 'YouthBks', 'CookBks', 'DoItYBks', 'RefBks',
    'ArtBks', 'GeogBks', 'ItalCook', 'ItalAtlas', 'ItalArt', 'Florence'
]
book_binary_matrix = (bookclub_data[book_columns] > 0).astype(int)
book_binary_matrix.head(10)


Qn 3.1: Charles Book Club binary incidence matrix (first 10 rows)


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence
0,0,1,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,0,1,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,1,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,1,0,0,0,0,0,0,0,0


In [17]:
print('Qn 3.2: Frequent itemsets for book purchases (min_support = 200/4000 = 0.05)')
book_frequent_itemsets = apriori(book_binary_matrix.astype(bool), min_support=0.05, use_colnames=True)
print(f'Number of frequent itemsets found: {len(book_frequent_itemsets)}')
book_frequent_itemsets.head(10)


Qn 3.2: Frequent itemsets for book purchases (min_support = 200/4000 = 0.05)
Number of frequent itemsets found: 61


,support,itemsets
0,0.39400,(ChildBks)
1,0.23825,(YouthBks)
2,0.41550,(CookBks)
3,0.25475,(DoItYBks)
4,0.20475,(RefBks)
5,0.22300,(ArtBks)
6,0.26675,(GeogBks)
7,0.10750,(ItalCook)
8,0.08450,(Florence)
9,0.14750,"(YouthBks, ChildBks)"


In [18]:
print('Qn 3.3: Top 25 association rules by lift (min_confidence = 0.5)')
book_rules = association_rules(book_frequent_itemsets, metric='confidence', min_threshold=0.5)
top_25_book_rules = (
    book_rules
    .sort_values(by='lift', ascending=False)
    .loc[:, ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']]
    .head(25)
    .reset_index(drop=True)
)
top_25_book_rules


Qn 3.3: Top 25 association rules by lift (min_confidence = 0.5)


,antecedents,consequents,support,confidence,lift,leverage
0,"(RefBks, YouthBks)","(CookBks, ChildBks)",0.05525,0.680000,2.809917,0.035588
1,"(DoItYBks, RefBks)","(CookBks, ChildBks)",0.06125,0.662162,2.736207,0.038865
2,"(DoItYBks, YouthBks)","(CookBks, ChildBks)",0.06700,0.648910,2.681448,0.042014
3,"(GeogBks, RefBks)","(CookBks, ChildBks)",0.05025,0.614679,2.539995,0.030467
4,"(GeogBks, YouthBks)","(CookBks, ChildBks)",0.06325,0.605263,2.501087,0.037961
5,"(DoItYBks, GeogBks)","(CookBks, ChildBks)",0.06050,0.599010,2.475248,0.036058
6,"(GeogBks, CookBks, ChildBks)",(YouthBks),0.06325,0.577626,2.424452,0.037162
7,"(CookBks, RefBks, ChildBks)",(DoItYBks),0.06125,0.591787,2.323013,0.034883
8,"(DoItYBks, GeogBks)",(YouthBks),0.05450,0.539604,2.264864,0.030437
9,"(CookBks, RefBks, ChildBks)",(YouthBks),0.05525,0.533816,2.240573,0.030591


In [19]:
print('Qn 4.1: Rule with highest support in book purchase rules')
q4_1_rule_highest_support = book_rules.sort_values(by='support', ascending=False).iloc[0]
q4_1_antecedents = q4_1_rule_highest_support['antecedents']
q4_1_consequents = q4_1_rule_highest_support['consequents']
q4_1_support = q4_1_rule_highest_support['support']
q4_1_confidence = q4_1_rule_highest_support['confidence']
q4_1_lift = q4_1_rule_highest_support['lift']
q4_1_result = pd.DataFrame([{
    'antecedents': q4_1_antecedents,
    'consequents': q4_1_consequents,
    'support': q4_1_support,
    'confidence': q4_1_confidence,
    'lift': q4_1_lift
}])
q4_1_result


Qn 4.1: Rule with highest support in book purchase rules


,antecedents,consequents,support,confidence,lift
0,(CookBks),(ChildBks),0.242,0.582431,1.478251


In [21]:
print('Qn 4.2: Highest-lift rule vs highest-support rule (trade-off)')
q4_2_rule_highest_lift = book_rules.sort_values(by='lift', ascending=False).iloc[0]
q4_2_highest_lift_support = q4_2_rule_highest_lift['support']
q4_2_highest_support_value = q4_1_support
q4_2_support_difference = q4_2_highest_support_value - q4_2_highest_lift_support
q4_2_comparison = pd.DataFrame([
    {
        'rule_type': 'highest_support_rule',
        'antecedents': q4_1_rule_highest_support['antecedents'],
        'consequents': q4_1_rule_highest_support['consequents'],
        'support': q4_2_highest_support_value,
        'confidence': q4_1_rule_highest_support['confidence'],
        'lift': q4_1_rule_highest_support['lift']
    },
    {
        'rule_type': 'highest_lift_rule',
        'antecedents': q4_2_rule_highest_lift['antecedents'],
        'consequents': q4_2_rule_highest_lift['consequents'],
        'support': q4_2_highest_lift_support,
        'confidence': q4_2_rule_highest_lift['confidence'],
        'lift': q4_2_rule_highest_lift['lift']
    }
])
q4_2_tradeoff_statement = (
    f"The highest-lift rule has support {q4_2_highest_lift_support:.4f}, while the highest-support rule has support {q4_2_highest_support_value:.4f}. "
    f"This shows the trade-off: very efficient rules (high lift) may apply to fewer transactions, while lower-lift rules can affect more transactions."
)
q4_2_comparison
print(q4_2_tradeoff_statement)


Qn 4.2: Highest-lift rule vs highest-support rule (trade-off)
The highest-lift rule has support 0.0553, while the highest-support rule has support 0.2420. This shows the trade-off: very efficient rules (high lift) may apply to fewer transactions, while lower-lift rules can affect more transactions.


In [20]:
print('Qn 4.3: Top 10 rules by lift and the one with lowest confidence')
q4_3_top_10_by_lift = book_rules.sort_values(by='lift', ascending=False).head(10).reset_index(drop=True)
q4_3_lowest_confidence_rule = q4_3_top_10_by_lift.sort_values(by='confidence', ascending=True).iloc[0]
q4_3_lowest_confidence_rule_df = pd.DataFrame([{
    'antecedents': q4_3_lowest_confidence_rule['antecedents'],
    'consequents': q4_3_lowest_confidence_rule['consequents'],
    'support': q4_3_lowest_confidence_rule['support'],
    'confidence': q4_3_lowest_confidence_rule['confidence'],
    'lift': q4_3_lowest_confidence_rule['lift'],
    'leverage': q4_3_lowest_confidence_rule['leverage']
}])
q4_3_top_10_by_lift[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']]
q4_3_lowest_confidence_rule_df


Qn 4.3: Top 10 rules by lift and the one with lowest confidence


,antecedents,consequents,support,confidence,lift,leverage
0,"(CookBks, RefBks, ChildBks)",(YouthBks),0.05525,0.533816,2.240573,0.030591


In [24]:
print('Qn 5.1: Synthetic random binary dataset (50 transactions x 9 items)')
random.seed(0)
q5_1_item_columns = [f'Item{i}' for i in range(1, 10)]
q5_1_synthetic_binary_matrix = pd.DataFrame(
    [[random.randint(0, 1) for _ in range(9)] for _ in range(50)],
    columns=q5_1_item_columns
)
q5_1_synthetic_binary_matrix.index = [f'T{idx}' for idx in range(1, 51)]
q5_1_synthetic_binary_matrix.head(10)


Qn 5.1: Synthetic random binary dataset (50 transactions x 9 items)


,Item1,Item2,Item3,Item4,Item5,Item6,Item7,Item8,Item9
T1,1,1,0,1,1,1,1,1,1
T2,0,0,1,0,0,1,0,1,0
T3,0,1,1,0,1,1,1,0,1
T4,1,1,0,0,0,1,0,1,1
T5,0,1,0,0,0,0,0,1,0
T6,0,1,1,0,1,1,0,1,0
T7,1,1,0,1,1,0,1,0,0
T8,0,0,1,1,0,0,0,0,0
T9,0,1,1,0,0,1,1,1,1
T10,1,0,1,0,1,1,0,0,0


In [25]:
print('Qn 5.2: Apriori (min_support = 0.04) and rules (min_confidence = 0.7)')
q5_2_min_support = 2 / 50
q5_2_frequent_itemsets = apriori(
    q5_1_synthetic_binary_matrix.astype(bool),
    min_support=q5_2_min_support,
    use_colnames=True
)
q5_2_rules = association_rules(
    q5_2_frequent_itemsets,
    metric='confidence',
    min_threshold=0.7
)
print(f'Frequent itemsets found: {len(q5_2_frequent_itemsets)}')
print(f'Association rules found: {len(q5_2_rules)}')
q5_2_frequent_itemsets.head(10)


Qn 5.2: Apriori (min_support = 0.04) and rules (min_confidence = 0.7)
Frequent itemsets found: 256
Association rules found: 179


,support,itemsets
0,0.56,(Item1)
1,0.64,(Item2)
2,0.40,(Item3)
3,0.48,(Item4)
4,0.42,(Item5)
5,0.64,(Item6)
6,0.44,(Item7)
7,0.38,(Item8)
8,0.42,(Item9)
9,0.36,"(Item1, Item2)"


In [26]:
print('Qn 5.3: Top 6 rules by uplift (lift) and interpretation')
q5_3_rules_with_uplift = q5_2_rules.copy()
q5_3_rules_with_uplift['uplift'] = q5_3_rules_with_uplift['lift']
q5_3_top_6_uplift_rules = (
    q5_3_rules_with_uplift
    .sort_values(by='uplift', ascending=False)
    .loc[:, ['antecedents', 'consequents', 'support', 'confidence', 'uplift']]
    .head(6)
    .reset_index(drop=True)
)
q5_3_max_uplift = q5_3_top_6_uplift_rules['uplift'].max() if len(q5_3_top_6_uplift_rules) > 0 else 0
q5_3_exceptional_uplift_found = bool(q5_3_max_uplift >= 2.0)
if q5_3_exceptional_uplift_found:
    q5_3_interpretation = (
        f"Yes. Even with random data, at least one rule has high uplift (max uplift = {q5_3_max_uplift:.3f}), which can happen by chance."
    )
else:
    q5_3_interpretation = (
        f"No exceptionally high uplift was observed (max uplift = {q5_3_max_uplift:.3f})."
    )
q5_3_top_6_uplift_rules
print(q5_3_interpretation)


Qn 5.3: Top 6 rules by uplift (lift) and interpretation
Yes. Even with random data, at least one rule has high uplift (max uplift = 7.143), which can happen by chance.


In [28]:
print('Qn 6.1: Synthetic ratings dataset (first 10 rows)')
np.random.seed(0)
q6_1_n_ratings = 5000
q6_1_ratings_data = pd.DataFrame({
    'itemID': np.random.randint(0, 100, size=q6_1_n_ratings),
    'userID': np.random.randint(0, 1000, size=q6_1_n_ratings),
    'rating': np.random.randint(1, 6, size=q6_1_n_ratings)
})
q6_1_ratings_data.head(10)


Qn 6.1: Synthetic ratings dataset (first 10 rows)


,itemID,userID,rating
0,44,187,3
1,47,507,3
2,64,493,2
3,67,183,1
4,67,893,3
5,9,673,4
6,83,267,3
7,21,639,1
8,36,987,2
9,87,802,1


In [30]:
# Qn 6.2
print('Qn 6.2: Convert to Surprise format and show train/test dimensions')
q6_2_surprise_df = q6_1_ratings_data[['userID', 'itemID', 'rating']].copy()
q6_2_surprise_df['userID'] = q6_2_surprise_df['userID'].astype(str)
q6_2_surprise_df['itemID'] = q6_2_surprise_df['itemID'].astype(str)
q6_2_reader = Reader(rating_scale=(1, 5))
q6_2_dataset = Dataset.load_from_df(q6_2_surprise_df, q6_2_reader)
q6_2_trainset, q6_2_testset = train_test_split(q6_2_dataset, test_size=0.2, random_state=0)
q6_2_train_size = q6_2_trainset.n_ratings
q6_2_test_size = len(q6_2_testset)
print(f'Train set size (ratings): {q6_2_train_size}')
print(f'Test set size (ratings): {q6_2_test_size}')


Qn 6.2: Convert to Surprise format and show train/test dimensions
Train set size (ratings): 4000
Test set size (ratings): 1000


In [31]:
print('Qn 6.3: User cosine similarity and item-based CF model')
q6_3_user_item_matrix = q6_1_ratings_data.pivot_table(
    index='userID', columns='itemID', values='rating', aggfunc='mean', fill_value=0
)
q6_3_user_vectors = q6_3_user_item_matrix.to_numpy(dtype=float)
q6_3_norms = np.linalg.norm(q6_3_user_vectors, axis=1, keepdims=True)
q6_3_norms[q6_3_norms == 0] = 1
q6_3_user_cosine_similarity = (q6_3_user_vectors @ q6_3_user_vectors.T) / (q6_3_norms @ q6_3_norms.T)
q6_3_user_cosine_similarity_df = pd.DataFrame(
    q6_3_user_cosine_similarity,
    index=q6_3_user_item_matrix.index,
    columns=q6_3_user_item_matrix.index
)
q6_3_sim_options = {'name': 'cosine', 'user_based': False}
q6_3_item_based_model = KNNBasic(sim_options=q6_3_sim_options, verbose=False)
q6_3_item_based_model.fit(q6_2_trainset)
q6_3_user_cosine_similarity_df.iloc[:10, :10]


Qn 6.3: User cosine similarity and item-based CF model


userID,0,1,2,3,4,5,6,7,8,9
userID,,,,,,,,,,
0,1.000000,0.0,0.444116,0.000000,0.0,0.00000,0.172559,0.391255,0.036980,0.000000
1,0.000000,1.0,0.000000,0.000000,0.0,0.00000,0.000000,0.000000,0.000000,0.000000
2,0.444116,0.0,1.000000,0.000000,0.0,0.32097,0.074720,0.457430,0.000000,0.302614
3,0.000000,0.0,0.000000,1.000000,0.0,0.00000,0.000000,0.000000,0.064676,0.061113
4,0.000000,0.0,0.000000,0.000000,1.0,0.00000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.0,0.320970,0.000000,0.0,1.00000,0.000000,0.000000,0.000000,0.000000
6,0.172559,0.0,0.074720,0.000000,0.0,0.00000,1.000000,0.131654,0.000000,0.000000
7,0.391255,0.0,0.457430,0.000000,0.0,0.00000,0.131654,1.000000,0.000000,0.000000
8,0.036980,0.0,0.000000,0.064676,0.0,0.00000,0.000000,0.000000,1.000000,0.151186


In [32]:

print('Qn 6.4: Predict unrated pairs and recommended items for each user')
# Recreate trainset/model if this cell is run independently
if 'q6_2_trainset' not in globals() or 'q6_3_item_based_model' not in globals():
    if 'q6_1_ratings_data' not in globals():
        np.random.seed(0)
        q6_1_n_ratings = 5000
        q6_1_ratings_data = pd.DataFrame({
            'itemID': np.random.randint(0, 100, size=q6_1_n_ratings),
            'userID': np.random.randint(0, 1000, size=q6_1_n_ratings),
            'rating': np.random.randint(1, 6, size=q6_1_n_ratings)
        })

    q6_2_surprise_df = q6_1_ratings_data[['userID', 'itemID', 'rating']].copy()
    q6_2_surprise_df['userID'] = q6_2_surprise_df['userID'].astype(str)
    q6_2_surprise_df['itemID'] = q6_2_surprise_df['itemID'].astype(str)
    q6_2_reader = Reader(rating_scale=(1, 5))
    q6_2_dataset = Dataset.load_from_df(q6_2_surprise_df, q6_2_reader)
    q6_2_trainset, q6_2_testset = train_test_split(q6_2_dataset, test_size=0.2, random_state=0)

    q6_3_sim_options = {'name': 'cosine', 'user_based': False}
    q6_3_item_based_model = KNNBasic(sim_options=q6_3_sim_options, verbose=False)
    q6_3_item_based_model.fit(q6_2_trainset)

q6_4_anti_testset = q6_2_trainset.build_anti_testset()
q6_4_predictions = q6_3_item_based_model.test(q6_4_anti_testset)
q6_4_top_n = 5
q6_4_user_recommendations = defaultdict(list)
for pred in q6_4_predictions:
    q6_4_user_recommendations[pred.uid].append((pred.iid, pred.est))
for uid in q6_4_user_recommendations:
    q6_4_user_recommendations[uid] = sorted(
        q6_4_user_recommendations[uid],
        key=lambda x: x[1],
        reverse=True
    )[:q6_4_top_n]
q6_4_recommendations_df = pd.DataFrame({
    'userID': list(q6_4_user_recommendations.keys()),
    'recommended_items': [
        [item for item, _ in q6_4_user_recommendations[u]] for u in q6_4_user_recommendations
    ]
}).sort_values(by="userID").reset_index(drop=True)
print(f'Predictions for unrated pairs: {len(q6_4_predictions)}')
q6_4_recommendations_df


Qn 6.4: Predict unrated pairs and recommended items for each user
Predictions for unrated pairs: 94264


,userID,recommended_items
0,0,"[50, 59, 72, 26, 12]"
1,10,"[67, 76, 19, 83, 22]"
2,100,"[82, 49, 55, 72, 44]"
3,101,"[46, 11, 51, 60, 20]"
4,102,"[12, 10, 65, 24, 8]"
...,...,...
977,995,"[31, 77, 99, 93, 36]"
978,996,"[76, 93, 44, 0, 65]"
979,997,"[27, 99, 86, 54, 55]"
980,998,"[76, 2, 38, 29, 52]"
